<a href="https://colab.research.google.com/github/alana-mch/Calorie-Tracker/blob/main/notebooks/ClalorieCalculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
#get train and test Food101 datasets
from torchvision import datasets

train_raw = datasets.Food101(root="data", split="train", download=True)
test_raw = datasets.Food101(root="data", split="test", download=True)

In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms

#transform images
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_food = datasets.Food101(root="data", split="train", transform=train_tf, download=True)
test_food = datasets.Food101(root="data", split="test", transform=test_tf, download=True)

In [ ]:
#training on non food
import random
from torch.utils.data import Dataset, ConcatDataset

#add a non-food category to classes
NOT_FOOD = 101
classes = train_food.classes + ["not_food"]
json.dump(classes, open("classes.json", "w"))

# non-food photos from Caltech101
caltech = datasets.Caltech101(root="data", download=True)

#categories in caltech that are food
banned = {"pizza", "strawberry", "lobster", "BACKGROUND_Google"}
allowed = [i for i, y in enumerate(caltech.y) if caltech.categories[y] not in banned]
#randomise caltech photos so they arent all from the same category
random.seed(0)
random.shuffle(allowed)
#use the same number of non food photos as the number of photos each category of food has
nf_train_idx, nf_test_idx = allowed[:750], allowed[750:1000]

#class for non_food photos that replaces caltechs label with our set NOT_FOOD label and transforms image
class NonFood(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.transform = base, indices, transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, _ = self.base[self.indices[i]]
        return self.transform(img.convert("RGB")), NOT_FOOD

train_data = ConcatDataset([train_food, NonFood(caltech, nf_train_idx, train_tf)])
test_data = ConcatDataset([test_food, NonFood(caltech, nf_test_idx, test_tf)])

In [ ]:
#create DataLoader objects in order to hand model batches of 32 images for faster processing
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

#run on external GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

#create model
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

#freeze all weights from being altered during training
for p in model.parameters():
    p.requires_grad = False

#replace 1 layer with a layer that produces 102 scores and these weights can be altered during training
model.classifier[1] = nn.Linear(1280, 102)

#moves model to chosen device
model = model.to(device)

In [ ]:
#create loss function and optimiser
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.classifier.parameters(), lr=0.001)

In [ ]:
#train
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        #predict
        scores = model(x)

        #measure error
        loss = loss_fn(scores, y)

        #weight adjustments
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        running_loss += loss.item()

In [ ]:
#improving model: let the last 3 blocks of the body learn too
for p in model.features[-3:].parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": model.features[-3:].parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-4},
])

best = 0
for epoch in range(4):
    model.train()
    running = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()
    acc = evaluate(test_loader)
    print(f"epoch {epoch+1}: loss {running/len(train_loader):.3f}, test acc {acc:.3f}")

    #pick best epoch so far
    if acc > best:
        best = acc
        torch.save(model.state_dict(), "food_model_finetuned.pth")

In [ ]:
import shutil
shutil.copy("food_model_finetuned.pth", "food_model.pth")

In [ ]:
#save weights
import json
torch.save(model.state_dict(), "food_model.pth")

In [ ]:
#run to reload model
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(1280, len(classes))
model.load_state_dict(torch.load("food_model.pth", map_location=device))
model = model.to(device)

In [ ]:
def evaluate(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return correct / total

model.load_state_dict(torch.load("food_model.pth", map_location=device))
model = model.to(device)
print(evaluate(test_loader))

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        preds = model(x).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(y.tolist())

# confusion matrix (counts of true vs predicted)
cm = confusion_matrix(all_labels, all_preds)

# per-class precision, recall and F1, plus overall accuracy at the bottom
print(classification_report(all_labels, all_preds, target_names=classes))

# the 10 most common mistakes
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)          # blank out the correct answers

pairs = [(cm_off[i, j], classes[i], classes[j])
         for i in range(len(classes)) for j in range(len(classes)) if cm_off[i, j] > 0]

for count, true_food, predicted in sorted(pairs, reverse=True)[:10]:
    print(f"{true_food} predicted as {predicted}: {count} times")

In [ ]:
#calorie dictionary
calories = {
    "apple_pie": 300, "baby_back_ribs": 500, "baklava": 300, "beef_carpaccio": 200,
    "beef_tartare": 250, "beet_salad": 180, "beignets": 300, "bibimbap": 560,
    "bread_pudding": 380, "breakfast_burrito": 450, "bruschetta": 200, "caesar_salad": 250,
    "cannoli": 250, "caprese_salad": 300, "carrot_cake": 450, "ceviche": 180,
    "cheesecake": 400, "cheese_plate": 450, "chicken_curry": 450, "chicken_quesadilla": 520,
    "chicken_wings": 430, "chocolate_cake": 420, "chocolate_mousse": 300, "churros": 340,
    "clam_chowder": 300, "club_sandwich": 590, "crab_cakes": 280, "creme_brulee": 340,
    "croque_madame": 600, "cup_cakes": 260, "deviled_eggs": 140, "donuts": 260,
    "dumplings": 300, "edamame": 190, "eggs_benedict": 600, "escargots": 230,
    "falafel": 330, "filet_mignon": 400, "fish_and_chips": 850, "foie_gras": 450,
    "french_fries": 365, "french_onion_soup": 370, "french_toast": 350, "fried_calamari": 400,
    "fried_rice": 330, "frozen_yogurt": 220, "garlic_bread": 300, "gnocchi": 350,
    "greek_salad": 250, "grilled_cheese_sandwich": 400, "grilled_salmon": 400, "guacamole": 230,
    "gyoza": 280, "hamburger": 500, "hot_and_sour_soup": 120, "hot_dog": 300,
    "huevos_rancheros": 500, "hummus": 210, "ice_cream": 270, "lasagna": 450,
    "lobster_bisque": 300, "lobster_roll_sandwich": 450, "macaroni_and_cheese": 500, "macarons": 200,
    "miso_soup": 60, "mussels": 280, "nachos": 600, "omelette": 300,
    "onion_rings": 400, "oysters": 100, "pad_thai": 650, "paella": 500,
    "pancakes": 350, "panna_cotta": 300, "peking_duck": 450, "pho": 450,
    "pizza": 285, "pork_chop": 350, "poutine": 740, "prime_rib": 700,
    "pulled_pork_sandwich": 500, "ramen": 500, "ravioli": 400, "red_velvet_cake": 450,
    "risotto": 450, "samosa": 300, "sashimi": 200, "scallops": 200,
    "seaweed_salad": 130, "shrimp_and_grits": 450, "spaghetti_bolognese": 500, "spaghetti_carbonara": 650,
    "spring_rolls": 300, "steak": 500, "strawberry_shortcake": 400, "sushi": 300,
    "tacos": 350, "takoyaki": 300, "tiramisu": 450, "tuna_tartare": 200,
    "waffles": 300,
}

import json
json.dump(calories, open("calories.json", "w"), indent=2)

In [ ]:
!pip install -q gradio

In [ ]:
import json
import torch
import torch.nn as nn
import gradio as gr
from torchvision import transforms
from torchvision.models import efficientnet_b0

device = "cpu"

#load files
classes = json.load(open("classes.json"))
calories = json.load(open("calories.json"))

#load model
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(1280, len(classes))
model.load_state_dict(torch.load("food_model.pth", map_location=device))
model.eval()

#transforms for images for the models
tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

THRESHOLD = 0.4
FLOOR = 0.2

def predict(img):
    if img is None:
        return {}, "Upload or take a photo of your meal."
    x = tf(img.convert("RGB")).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]

    confidences = {classes[i]: float(probs[i]) for i in range(len(classes))}

    top = probs.topk(4)
    names = [classes[i] for i in top.indices.tolist()]
    p = top.values[0].item()
    food = names[0]
    alternatives = [n for n in names[1:] if n != "not_food"][:2]

    if food == "not_food":
        message = "This doesn't look like food I recognise."
    elif p < FLOOR:
        message = "Not sure what this is."
    elif p < THRESHOLD:
        message = (f"Best guess: {food} (about {calories[food]} kcal), "
                   f"but it could also be {' or '.join(alternatives)}.")
    else:
        message = f"{food}: about {calories[food]} kcal per typical serving"
    return confidences, message

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", sources=["upload", "webcam"], label="Photo of your meal"),
    outputs=[gr.Label(num_top_classes=3, label="Prediction"),
             gr.Textbox(label="Calorie estimate")],
    title="Meal Scanner",
    description=("Recognises 101 foods from the Food-101 dataset and gives an approximate "
                 "calorie estimate for a typical serving. One food per photo; portion size "
                 "is not estimated."),
)

demo.launch()